# Embeddings and multi-input models — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D020_02_Multi_Input_Models.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# Three Input Models with Keras Functional API

## Introduction
In the previous notebook, we built a model with two inputs (Team 1 and Team 2) that shared an embedding layer. Now, we will make our model more realistic by adding a **third input**: whether a team is playing at home or away.

This introduces the concept of **Mixed Data Types** in a neural network:
1.  **Categorical Inputs**: Team IDs (handled via Embeddings).
2.  **Numerical/Binary Inputs**: Home advantage (handled directly or sufficiently simple to stay numeric).

The **Keras Functional API** shines here, allowing us to build a graph where different inputs enter at different points and merge together.

### Objectives
- Load and preprocess mixed data (categorical + numeric).
- Build a 3-input model using Keras Functional API.
- Understand how to merge (Concatenate) embeddings with raw features.

---

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import plot_model
import matplotlib.pyplot as plt

# Set seed
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# ═══ CHECKPOINT — "Preparing the Dataset" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import plot_model
import matplotlib.pyplot as plt

print("Checkpoint restored. Ready for: Preparing the Dataset")

## 1. Preparing the Dataset
We use the same `games_season.csv` dataset.

In [ ]:
# TODO: compute `df`
# Name your result: df

### Preprocessing Inputs
1.  **Team IDs**: Convert to integer range `[0, N-1]` for the embedding layer.
2.  **Home Column**: The dataset has a `home` column. Let's look at it.

In [ ]:
# TODO: compute `input_1`, `input_2`, `input_3` from `team_1`, `team_2`, `home`
# Name your result: input_1, input_2, input_3, target

---
## 2. Building the 3-Input Model

### Architecture Diagram Concept
We will have 3 branches:
- **Branch 1**: Team 1 Input -> Embedding -> Flatten
- **Branch 2**: Team 2 Input -> Embedding -> Flatten
- **Branch 3**: Home Input (Numeric) -> Input Layer

Then we **Concatenate** all three vectors.

In [ ]:
# TODO: compute `embed_dim`, `in_team_1`, `in_team_2`
# Name your result: embed_dim, in_team_1, in_team_2, in_home

### Architecture Visualization
This plot clarifies the multi-input structure.

In [ ]:
try:
    plot_model(model_3, show_shapes=True)
except Exception as e:
    print('plot_model needs graphviz:', e)

In [ ]:
# TODO: compute `history`, `epochs`, `batch_size`
# Name your result: history, epochs, batch_size, validation_split

---
## 3. Exercises

### Exercise 1: Adding a 4th Input
Imagine we want to add the **Season** year (`season` column) as a feature.
Since year is numerical (e.g., 1985, 2000), you could treat it as a number (`shape=(1,)`) or as a category (Embedding).

Task: Modify the architecture to accept `season` as a **numerical input** (Input 4). Concatenate it with the others.

In [ ]:
# Exercise 1 Solution: Adding a 4th Input (Season)
input_4 = df['season']  # Numerical input

# Hyperparameters
embed_dim = 8

# --- Inputs (now 4) ---
in_team_1_4 = layers.Input(shape=(1,), name='Team1_ID_4')
in_team_2_4 = layers.Input(shape=(1,), name='Team2_ID_4')
in_home_4   = layers.Input(shape=(1,), name='Home_Advantage_4')
in_season   = layers.Input(shape=(1,), name='Season')  # NEW: 4th input

# --- Embedding Component (Shared) ---
embed_layer_4 = layers.Embedding(input_dim=num_unique_teams, 
                                  output_dim=embed_dim, 
                                  input_length=1, 
                                  name='Team_Embedding_4')

# Lookup process
t1_embed_4 = embed_layer_4(in_team_1_4)
t2_embed_4 = embed_layer_4(in_team_2_4)

# Flatten embeddings
t1_flat_4 = layers.Flatten()(t1_embed_4)
t2_flat_4 = layers.Flatten()(t2_embed_4)

# --- Merging Layers (now 4 inputs) ---
# Concatenate: 8 + 8 + 1 + 1 = 18 features
merged_4 = layers.Concatenate()([t1_flat_4, t2_flat_4, in_home_4, in_season])

# --- Dense Layers ---
x_4 = layers.Dense(64, activation='relu')(merged_4)
x_4 = layers.Dropout(0.2)(x_4)
output_4 = layers.Dense(1, name='Score_Diff_4')(x_4)

# --- Final Model ---
model_4 = models.Model(inputs=[in_team_1_4, in_team_2_4, in_home_4, in_season], outputs=output_4)
model_4.summary()

# Train with 4 inputs
model_4.compile(optimizer='adam', loss='mae')
# history_4 = model_4.fit([input_1_int, input_2_int, input_3, input_4], target, epochs=3, batch_size=2048, validation_split=0.1)

In [ ]:
# ═══ CHECKPOINT — "Exercise 2: Compare Models" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import plot_model
import matplotlib.pyplot as plt
df = pd.read_csv('../../00-data/games_season.csv')
input_1 = df['team_1']
input_2 = df['team_2']
input_3 = df['home']
target = df['score_diff']
unique_teams = np.unique(np.concatenate([input_1.unique(), input_2.unique()]))
num_unique_teams = len(unique_teams)
team_to_int = {team: i for i, team in enumerate(unique_teams)}
input_1_int = input_1.map(team_to_int)
input_2_int = input_2.map(team_to_int)
embed_dim = 8
in_team_1 = layers.Input(shape=(1,), name='Team1_ID')
in_team_2 = layers.Input(shape=(1,), name='Team2_ID')
in_home = layers.Input(shape=(1,), name='Home_Advantage')
embed_layer = layers.Embedding(input_dim=num_unique_teams, output_dim=embed_dim, input_length=1, name='Team_Embedding')
t1_embed = embed_layer(in_team_1)
t2_embed = embed_layer(in_team_2)
t1_flat = layers.Flatten()(t1_embed)
t2_flat = layers.Flatten()(t2_embed)
merged = layers.Concatenate()([t1_flat, t2_flat, in_home])
x = layers.Dense(64, activation='relu')(merged)
x = layers.Dropout(0.2)(x)
output = layers.Dense(1, name='Score_Diff')(x)
model_3 = models.Model(inputs=[in_team_1, in_team_2, in_home], outputs=output)
model_3.compile(optimizer='adam', loss='mae')
history = model_3.fit([input_1_int, input_2_int, input_3], target, epochs=1, batch_size=2048, validation_split=0.1)
input_4 = df['season']
embed_dim = 8
in_team_1_4 = layers.Input(shape=(1,), name='Team1_ID_4')
in_team_2_4 = layers.Input(shape=(1,), name='Team2_ID_4')
in_home_4 = layers.Input(shape=(1,), name='Home_Advantage_4')
in_season = layers.Input(shape=(1,), name='Season')
embed_layer_4 = layers.Embedding(input_dim=num_unique_teams, output_dim=embed_dim, input_length=1, name='Team_Embedding_4')
t1_embed_4 = embed_layer_4(in_team_1_4)
t2_embed_4 = embed_layer_4(in_team_2_4)
t1_flat_4 = layers.Flatten()(t1_embed_4)
t2_flat_4 = layers.Flatten()(t2_embed_4)
merged_4 = layers.Concatenate()([t1_flat_4, t2_flat_4, in_home_4, in_season])
x_4 = layers.Dense(64, activation='relu')(merged_4)
x_4 = layers.Dropout(0.2)(x_4)
output_4 = layers.Dense(1, name='Score_Diff_4')(x_4)
model_4 = models.Model(inputs=[in_team_1_4, in_team_2_4, in_home_4, in_season], outputs=output_4)
model_4.compile(optimizer='adam', loss='mae')

print("Checkpoint restored. Ready for: Exercise 2: Compare Models")

### Exercise 2: Compare Models
Evaluate this 3-input model against the 2-input model from the previous notebook. 
Does knowing Home/Away improve the Mean Absolute Error (MAE) loss?

*Hint: Train both for the same number of epochs and compare the final validation loss.*